In [1]:
import sys
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.utils.data as data
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim

sys.path.append(os.path.abspath('..'))

from src.config import TARGET_CLASSES, IMAGE_SIZE, BATCH_SIZE, GENERATE_COUNT_MAP
from src.utils.data_loader import ButterflyDataset
from src.utils.metrics import inception_score, frechet_inception_distance, mean_ssim

NUM_CLASSES = len(TARGET_CLASSES)   # número de classes condicionais
print(f"Classes: {TARGET_CLASSES}")
print(f"NUM_CLASSES: {NUM_CLASSES}")


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Classes: ['AMERICAN SNOOT', 'CRIMSON PATCH', 'MALACHITE', 'GOLD BANDED', 'WOOD SATYR']
NUM_CLASSES: 5


In [2]:
path    = "splits"
img_dir = '../data/raw/train'
df      = pd.read_csv(os.path.join(path, 'full_train_split.csv'))

data_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),          # augmentação leve — ajuda o G
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],       # [0,1] → [-1, 1]  (alinha com Tanh)
                         [0.5, 0.5, 0.5]),
])

# target_classes=None → usa todas as classes do dataset para o treino
dataset    = ButterflyDataset(df=df, img_dir=img_dir,
                               transform=data_transform,
                               target_classes=None)
dataloader = data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"Dataset: {len(dataset)} imagens | {len(dataloader)} batches/época")


Dataset: 4159 imagens | 130 batches/época


In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available()       else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)
print("Device:", device)

Device: mps


In [4]:
def show_images(imgs, nrow=4):
    fig, axes = plt.subplots(nrow, nrow, figsize=(8, 8))
    for i, ax in enumerate(axes.flatten()):
        img = imgs[i].permute(1, 2, 0).numpy()
        ax.imshow(img.clip(0, 1))
        ax.axis('off')
    plt.tight_layout()
    plt.show()


def weights_init(m):
    """Inicialização DCGAN recomendada pelo paper original."""
    cname = m.__class__.__name__
    if "Conv" in cname or "Linear" in cname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in cname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Modelo 1 — cDCGAN (Conditional DCGAN)

In [5]:
class Generator(nn.Module):
    """Conditional DCGAN Generator: (z + class_emb) → imagem RGB (3×64×64)."""
    def __init__(self, nz=100, ngf=64, nc=3, num_classes=NUM_CLASSES, emb_dim=64):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, emb_dim)
        in_dim = nz + emb_dim
        self.main = nn.Sequential(
            # (nz+emb_dim) → 4×4
            nn.ConvTranspose2d(in_dim, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            # 4×4 → 8×8
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            # 8×8 → 16×16
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            # 16×16 → 32×32
            nn.ConvTranspose2d(ngf*2, ngf,   4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),   nn.ReLU(True),
            # 32×32 → 64×64
            nn.ConvTranspose2d(ngf,   nc,    4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z, labels):
        # z: (B, nz, 1, 1) | labels: (B,)
        emb = self.label_emb(labels).unsqueeze(-1).unsqueeze(-1)  # (B, emb_dim, 1, 1)
        inp = torch.cat([z, emb], dim=1)                          # (B, nz+emb_dim, 1, 1)
        return self.main(inp)


In [6]:
class Discriminator(nn.Module):
    """Conditional DCGAN Discriminator: (img + projected class) → probabilidade."""
    def __init__(self, nc=3, ndf=64, num_classes=NUM_CLASSES, emb_dim=64):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, emb_dim)
        self.proj      = nn.Linear(emb_dim, IMAGE_SIZE * IMAGE_SIZE)
        # O discriminador recebe a imagem + 1 canal de projecção de label → nc+1 canais
        self.model = nn.Sequential(
            nn.Conv2d(nc + 1, ndf,   4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf,   ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1,     4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img, labels):
        # Projectar label para mapa espacial e concatenar com imagem
        emb  = self.label_emb(labels)                              # (B, emb_dim)
        lmap = self.proj(emb).view(-1, 1, IMAGE_SIZE, IMAGE_SIZE)  # (B, 1, H, W)
        inp  = torch.cat([img, lmap], dim=1)                       # (B, nc+1, H, W)
        return self.model(inp).view(-1)


In [7]:
NZ1 = 100

netG1 = Generator(nz=NZ1).to(device)
netD1 = Discriminator().to(device)
netG1.apply(weights_init)
netD1.apply(weights_init)

optimizerD1 = optim.Adam(netD1.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizerG1 = optim.Adam(netG1.parameters(), lr=2e-4, betas=(0.5, 0.999))

print("Generator1     params:", sum(p.numel() for p in netG1.parameters()))
print("Discriminator1 params:", sum(p.numel() for p in netD1.parameters()))


Generator1     params: 4101312
Discriminator1 params: 3033152


## Treino Modelo 1

In [15]:
criterion1  = nn.BCELoss()
real_label  = 1
fake_label  = 0
EPOCHS1     = 200

os.makedirs("checkpoints", exist_ok=True)
G1_losses, D1_losses = [], []

for epoch in range(EPOCHS1):
    netG1.train(); netD1.train()
    epoch_G = epoch_D = 0.0

    for i, (imgs, labels) in enumerate(dataloader):
        bs       = imgs.size(0)
        real_cpu = imgs.to(device)
        labels   = labels.to(device)

        lbl_real = torch.full((bs,), real_label, dtype=torch.float, device=device)
        lbl_fake = torch.full((bs,), fake_label, dtype=torch.float, device=device)

        # ── Discriminador ─────────────────────────────────────────────────
        netD1.zero_grad()
        errD_real = criterion1(netD1(real_cpu, labels), lbl_real)
        noise     = torch.randn(bs, NZ1, 1, 1, device=device)
        fake_lbls = torch.randint(0, NUM_CLASSES, (bs,), device=device)
        fake      = netG1(noise, fake_lbls)
        errD_fake = criterion1(netD1(fake.detach(), fake_lbls), lbl_fake)
        errD      = errD_real + errD_fake
        errD.backward()
        optimizerD1.step()

        # ── Gerador — 2 updates por batch ─────────────────────────────────
        for _ in range(2):
            netG1.zero_grad()
            noise_g     = torch.randn(bs, NZ1, 1, 1, device=device)
            fake_lbls_g = torch.randint(0, NUM_CLASSES, (bs,), device=device)
            fake_g      = netG1(noise_g, fake_lbls_g)
            target_ones = torch.ones(bs, device=device)  # ← 1.0, não 0.9
            errG        = criterion1(netD1(fake_g, fake_lbls_g), target_ones)
            errG.backward()
            optimizerG1.step()

        epoch_G += errG.item()
        epoch_D += errD.item()

        # if i % 100 == 0:
        #     print(f"[{epoch}/{EPOCHS1}]"
        #           f"  D: {errD.item():.4f}  G: {errG.item():.4f}")
        
    print(f"Época [{epoch+1}/{EPOCHS1}]  D: {errD.item():.4f}  G: {errG.item():.4f}")

    avg_G1 = epoch_G / len(dataloader)
    avg_D1 = epoch_D / len(dataloader)
    G1_losses.append(avg_G1); D1_losses.append(avg_D1)

    # if epoch % 10 == 0:
    #     with torch.no_grad():
    #         sample_z   = torch.randn(16, NZ1, 1, 1, device=device)
    #         sample_lbl = torch.arange(NUM_CLASSES, device=device).repeat_interleave(16 // NUM_CLASSES + 1)[:16]
    #         vis = (netG1(sample_z, sample_lbl).cpu() + 1) * 0.5
    #     show_images(vis, nrow=4)

    if epoch % 20 == 0:
        torch.save(netG1.state_dict(), f"checkpoints/netG1_epoch{epoch:03d}.pt")
        torch.save(netD1.state_dict(), f"checkpoints/netD1_epoch{epoch:03d}.pt")

print("Treino Modelo 1 concluído.")
torch.save(netG1.state_dict(), "checkpoints/netG1_final.pt")


KeyboardInterrupt: 

# Modelo 2 — cLSGAN (Conditional Least Squares GAN)

LSGAN substitui a loss BCE por **MSE**, resolvendo vanishing gradients
e produzindo imagens mais nítidas. A versão condicional injeta o label
tanto no Generator (via embedding) como no Discriminator (via projecção espacial).


In [ ]:
class GeneratorLS(nn.Module):
    """Conditional LSGAN Generator."""
    def __init__(self, nz=100, ngf=64, nc=3, num_classes=NUM_CLASSES, emb_dim=64):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, emb_dim)
        in_dim = nz + emb_dim
        self.main = nn.Sequential(
            nn.ConvTranspose2d(in_dim, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf,   4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),   nn.ReLU(True),
            nn.ConvTranspose2d(ngf,   nc,    4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z, labels):
        emb = self.label_emb(labels).unsqueeze(-1).unsqueeze(-1)
        inp = torch.cat([z, emb], dim=1)
        return self.main(inp)


class DiscriminatorLS(nn.Module):
    """Conditional LSGAN Discriminator — sem Sigmoid (regressão MSE)."""
    def __init__(self, nc=3, ndf=64, num_classes=NUM_CLASSES, emb_dim=64):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, emb_dim)
        self.proj      = nn.Linear(emb_dim, IMAGE_SIZE * IMAGE_SIZE)
        self.model = nn.Sequential(
            nn.Conv2d(nc + 1, ndf,   4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf,   ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1,     4, 1, 0, bias=False)
            # SEM Sigmoid
        )

    def forward(self, img, labels):
        emb  = self.label_emb(labels)
        lmap = self.proj(emb).view(-1, 1, IMAGE_SIZE, IMAGE_SIZE)
        inp  = torch.cat([img, lmap], dim=1)
        return self.model(inp).view(-1)


In [ ]:
NZ2 = 100

netG2 = GeneratorLS(nz=NZ2).to(device)
netD2 = DiscriminatorLS().to(device)
netG2.apply(weights_init)
netD2.apply(weights_init)

optimizerD2 = optim.Adam(netD2.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizerG2 = optim.Adam(netG2.parameters(), lr=2e-4, betas=(0.5, 0.999))

print("Generator2 (cLSGAN)     params:", sum(p.numel() for p in netG2.parameters()))
print("Discriminator2 (cLSGAN) params:", sum(p.numel() for p in netD2.parameters()))


## Treino Modelo 2

In [ ]:
mse_loss = nn.MSELoss()
EPOCHS2  = 200

G2_losses, D2_losses = [], []

for epoch in range(EPOCHS2):
    netG2.train(); netD2.train()
    epoch_G = epoch_D = 0.0

    for i, (imgs, labels) in enumerate(dataloader):
        bs       = imgs.size(0)
        real_cpu = imgs.to(device)
        labels   = labels.to(device)

        ones  = torch.ones(bs,  device=device)
        zeros = torch.zeros(bs, device=device)

        # ── Discriminador ─────────────────────────────────────────────────
        netD2.zero_grad()
        errD_real = mse_loss(netD2(real_cpu, labels), ones)
        noise     = torch.randn(bs, NZ2, 1, 1, device=device)
        fake_lbls = torch.randint(0, NUM_CLASSES, (bs,), device=device)
        fake      = netG2(noise, fake_lbls)
        errD_fake = mse_loss(netD2(fake.detach(), fake_lbls), zeros)
        errD      = 0.5 * (errD_real + errD_fake)
        errD.backward()
        optimizerD2.step()

        # ── Gerador — 2 updates por batch ─────────────────────────────────
        for _ in range(2):
            netG2.zero_grad()
            noise_g     = torch.randn(bs, NZ2, 1, 1, device=device)
            fake_lbls_g = torch.randint(0, NUM_CLASSES, (bs,), device=device)
            fake_g      = netG2(noise_g, fake_lbls_g)
            errG        = 0.5 * mse_loss(netD2(fake_g, fake_lbls_g), ones)
            errG.backward()
            optimizerG2.step()

        epoch_G += errG.item()
        epoch_D += errD.item()

        if i % 100 == 0:
            print(f"[{epoch}/{EPOCHS2}][{i}/{len(dataloader)}]"
                  f"  D: {errD.item():.4f}  G: {errG.item():.4f}")

    avg_G2 = epoch_G / len(dataloader)
    avg_D2 = epoch_D / len(dataloader)
    G2_losses.append(avg_G2); D2_losses.append(avg_D2)

    if epoch % 10 == 0:
        with torch.no_grad():
            sample_z   = torch.randn(16, NZ2, 1, 1, device=device)
            sample_lbl = torch.arange(NUM_CLASSES, device=device).repeat_interleave(16 // NUM_CLASSES + 1)[:16]
            vis = (netG2(sample_z, sample_lbl).cpu() + 1) * 0.5
        show_images(vis, nrow=4)

    if epoch % 20 == 0:
        torch.save(netG2.state_dict(), f"checkpoints/netG2_epoch{epoch:03d}.pt")
        torch.save(netD2.state_dict(), f"checkpoints/netD2_epoch{epoch:03d}.pt")

print("Treino Modelo 2 concluído.")
torch.save(netG2.state_dict(), "checkpoints/netG2_final.pt")


# Modelo 3 — cWGAN-GP (Conditional Wasserstein GAN com Gradient Penalty)

Versão condicional do WGAN-GP. O Critic recebe a projecção espacial do label
concatenada com a imagem, e o Generator recebe o embedding de label concatenado com z.


In [ ]:
class Generator3(nn.Module):
    """Conditional WGAN-GP Generator."""
    def __init__(self, nz=100, ngf=64, nc=3, num_classes=NUM_CLASSES, emb_dim=64):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, emb_dim)
        in_dim = nz + emb_dim
        self.main = nn.Sequential(
            nn.ConvTranspose2d(in_dim, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf,   4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),   nn.ReLU(True),
            nn.ConvTranspose2d(ngf,   nc,    4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z, labels):
        emb = self.label_emb(labels).unsqueeze(-1).unsqueeze(-1)
        inp = torch.cat([z, emb], dim=1)
        return self.main(inp)


class Critic(nn.Module):
    """Conditional WGAN-GP Critic — InstanceNorm, sem Sigmoid."""
    def __init__(self, nc=3, ndf=64, num_classes=NUM_CLASSES, emb_dim=64):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, emb_dim)
        self.proj      = nn.Linear(emb_dim, IMAGE_SIZE * IMAGE_SIZE)
        self.model = nn.Sequential(
            nn.Conv2d(nc + 1, ndf,   4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf,   ndf*2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf*2, affine=True), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf*4, affine=True), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf*8, affine=True), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*8, 1,     4, 1, 0, bias=False)
        )

    def forward(self, img, labels):
        emb  = self.label_emb(labels)
        lmap = self.proj(emb).view(-1, 1, IMAGE_SIZE, IMAGE_SIZE)
        inp  = torch.cat([img, lmap], dim=1)
        return self.model(inp).view(-1)


def gradient_penalty(critic, real, fake, labels, device):
    """Gradient Penalty WGAN-GP — condicional (passa labels ao Critic)."""
    B     = real.size(0)
    alpha = torch.rand(B, 1, 1, 1, device=device)
    interp    = (alpha * real.detach() + (1 - alpha) * fake.detach()).requires_grad_(True)
    d_interp  = critic(interp, labels)
    grads     = torch.autograd.grad(
        outputs=d_interp,
        inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]
    grads = grads.view(B, -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


In [ ]:
NZ3       = 100
LAMBDA_GP = 10
N_CRITIC  = 5

netG3 = Generator3(nz=NZ3).to(device)
netC3 = Critic().to(device)
netG3.apply(weights_init)
netC3.apply(weights_init)

optimizerG3 = optim.Adam(netG3.parameters(), lr=1e-4, betas=(0.0, 0.9))
optimizerC3 = optim.Adam(netC3.parameters(), lr=1e-4, betas=(0.0, 0.9))

print("Generator3 params:", sum(p.numel() for p in netG3.parameters()))
print("Critic3    params:", sum(p.numel() for p in netC3.parameters()))


## Treino Modelo 3

In [ ]:
EPOCHS3   = 200

G3_losses, W3_dists = [], []

for epoch in range(EPOCHS3):
    netG3.train(); netC3.train()
    epoch_G = epoch_W = 0.0

    for i, (imgs, labels) in enumerate(dataloader):
        bs        = imgs.size(0)
        real_imgs = imgs.to(device)
        labels    = labels.to(device)

        # ── Critic (N_CRITIC passos) ───────────────────────────────────────
        for _ in range(N_CRITIC):
            netC3.zero_grad()
            fake_lbls = torch.randint(0, NUM_CLASSES, (bs,), device=device)
            z         = torch.randn(bs, NZ3, 1, 1, device=device)
            fake      = netG3(z, fake_lbls).detach()
            gp        = gradient_penalty(netC3, real_imgs, fake, labels, device)
            lossC     = (-netC3(real_imgs, labels).mean()
                         + netC3(fake, fake_lbls).mean()
                         + LAMBDA_GP * gp)
            lossC.backward()
            optimizerC3.step()

        # ── Gerador (1 passo) ─────────────────────────────────────────────
        netG3.zero_grad()
        fake_lbls_g = torch.randint(0, NUM_CLASSES, (bs,), device=device)
        z_g         = torch.randn(bs, NZ3, 1, 1, device=device)
        fake_g      = netG3(z_g, fake_lbls_g)
        lossG       = -netC3(fake_g, fake_lbls_g).mean()
        lossG.backward()
        optimizerG3.step()

        with torch.no_grad():
            w_dist = netC3(real_imgs, labels).mean() - netC3(fake_g.detach(), fake_lbls_g).mean()

        epoch_G += lossG.item()
        epoch_W += w_dist.item()

        if i % 100 == 0:
            print(f"[{epoch}/{EPOCHS3}][{i}/{len(dataloader)}]"
                  f"  W_dist: {w_dist.item():.4f}  G: {lossG.item():.4f}")

    avg_G3 = epoch_G / len(dataloader)
    avg_W3 = epoch_W / len(dataloader)
    G3_losses.append(avg_G3); W3_dists.append(avg_W3)

    if epoch % 10 == 0:
        with torch.no_grad():
            sample_z   = torch.randn(16, NZ3, 1, 1, device=device)
            sample_lbl = torch.arange(NUM_CLASSES, device=device).repeat_interleave(16 // NUM_CLASSES + 1)[:16]
            vis = (netG3(sample_z, sample_lbl).cpu() + 1) * 0.5
        show_images(vis, nrow=4)

    if epoch % 20 == 0:
        torch.save(netG3.state_dict(), f"checkpoints/netG3_epoch{epoch:03d}.pt")
        torch.save(netC3.state_dict(), f"checkpoints/netC3_epoch{epoch:03d}.pt")

print("Treino Modelo 3 concluído.")
torch.save(netG3.state_dict(), "checkpoints/netG3_final.pt")


# Geração de Imagens

In [13]:
import os
from torchvision.utils import save_image

OUTPUT_DIR = '../data/augmented'

def generate_and_save(generator, generate_count_map, nz, device,
                      target_classes, output_dir=OUTPUT_DIR, model_name="model"):
    """Gera imagens condicionais por classe e guarda em data/augmented/<class_name>/.

    Args:
        generator      : modelo Generator treinado (condicional)
        generate_count_map : dict {class_idx: n_images}  (de config.py)
        nz             : dimensão do ruído latente
        device         : torch device
        target_classes : lista de nomes de classe (TARGET_CLASSES)
        output_dir     : directório raiz de saída
        model_name     : prefixo do ficheiro (e.g. 'dcgan', 'lsgan', 'wgangp')
    Returns:
        list of tensors (C,H,W) em [0,1]
    """
    generator.eval()
    all_generated = []

    with torch.no_grad():
        for class_idx, num_to_gen in generate_count_map.items():
            class_name = target_classes[class_idx]
            save_dir   = os.path.join(output_dir, class_name)
            os.makedirs(save_dir, exist_ok=True)

            labels = torch.full((num_to_gen,), class_idx, dtype=torch.long, device=device)
            z      = torch.randn(num_to_gen, nz, 1, 1, device=device)
            imgs   = generator(z, labels)           # [-1, 1]
            imgs01 = (imgs.cpu() + 1) * 0.5         # → [0, 1]

            for j, img in enumerate(imgs01):
                fname = os.path.join(save_dir, f"{model_name}_{j:04d}.png")
                save_image(img, fname)

            print(f"  [{class_name}] {num_to_gen} imagens guardadas em {save_dir}")
            all_generated.append(imgs01)

    return list(torch.cat(all_generated, dim=0))


total_expected = sum(GENERATE_COUNT_MAP.values())
print(f"Total de imagens a gerar (segundo config): {total_expected}")
print(f"Destino: {OUTPUT_DIR}/<class_name>/")
print()


Total de imagens a gerar (segundo config): 235
Destino: ../data/augmented/<class_name>/



In [14]:

print("Gerando com Modelo 1 (cDCGAN)...")
fake_imgs_m1 = generate_and_save(netG1, GENERATE_COUNT_MAP, NZ1, device,
                                  TARGET_CLASSES, model_name="dcgan")

# print("\nGerando com Modelo 2 (cLSGAN)...")
# fake_imgs_m2 = generate_and_save(netG2, GENERATE_COUNT_MAP, NZ2, device,
#                                   TARGET_CLASSES, model_name="lsgan")

# print("\nGerando com Modelo 3 (cWGAN-GP)...")
# fake_imgs_m3 = generate_and_save(netG3, GENERATE_COUNT_MAP, NZ3, device,
#                                   TARGET_CLASSES, model_name="wgangp")

# print(f"\nGeradas: M1={len(fake_imgs_m1)} | M2={len(fake_imgs_m2)} | M3={len(fake_imgs_m3)}")
# print(f"Todas as imagens guardadas em '{OUTPUT_DIR}/'")

Gerando com Modelo 1 (cDCGAN)...
  [AMERICAN SNOOT] 46 imagens guardadas em ../data/augmented/AMERICAN SNOOT
  [CRIMSON PATCH] 47 imagens guardadas em ../data/augmented/CRIMSON PATCH
  [MALACHITE] 47 imagens guardadas em ../data/augmented/MALACHITE
  [GOLD BANDED] 47 imagens guardadas em ../data/augmented/GOLD BANDED
  [WOOD SATYR] 48 imagens guardadas em ../data/augmented/WOOD SATYR


In [ ]:
# Recolher imagens reais para FID e SSIM
print("Recolhendo imagens reais do dataloader...")
real_imgs_list = []
with torch.no_grad():
    for imgs, _ in dataloader:
        real_imgs_list.append(imgs)  # ToTensor() já dá [0,1]
        if sum(x.size(0) for x in real_imgs_list) >= total_expected:
            break

real_imgs_all = list(torch.cat(real_imgs_list, dim=0)[:total_expected])
print(f"Imagens reais recolhidas: {len(real_imgs_all)}")

# Métricas de Avaliação

| Métrica | Significado | Melhor valor |
|---------|-------------|--------------|
| **IS** | Qualidade + diversidade das imagens geradas | ↑ maior |
| **FID** | Distância entre distribuição real e gerada | ↓ menor |
| **SSIM** | Similaridade estrutural pixel-a-pixel | ↑ maior |


In [ ]:
metrics_device = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 58)
print("         AVALIAÇÃO COMPARATIVA — 3 MODELOS GAN")
print("=" * 58)

results = {}

for name, fake_list in [("Modelo 1 — DCGAN",   fake_imgs_m1),
                         ("Modelo 2 — LSGAN",   fake_imgs_m2),
                         ("Modelo 3 — WGAN-GP", fake_imgs_m3)]:
    print(f"\n>>> {name}")

    is_mean, is_std = inception_score(
        fake_list, batch_size=32, splits=5, device=metrics_device
    )
    fid_val  = frechet_inception_distance(
        real_imgs_all, fake_list, batch_size=32, device=metrics_device
    )
    ssim_val = mean_ssim(real_imgs_all, fake_list)

    print(f"  IS   (↑ melhor): {is_mean:.4f} ± {is_std:.4f}")
    print(f"  FID  (↓ melhor): {fid_val:.4f}")
    print(f"  SSIM (↑ melhor): {ssim_val:.4f}")

    results[name] = {"IS_mean": is_mean, "IS_std": is_std,
                     "FID": fid_val, "SSIM": ssim_val}

print("\n" + "=" * 58)
print(f"{'Modelo':<25} {'IS':>10} {'FID':>10} {'SSIM':>10}")
print("-" * 58)
for mname, m in results.items():
    print(f"{mname:<25} {m['IS_mean']:>10.4f} {m['FID']:>10.4f} {m['SSIM']:>10.4f}")
print("=" * 58)

In [ ]:
from collections import Counter

best_is   = max(results, key=lambda k: results[k]["IS_mean"])
best_fid  = min(results, key=lambda k: results[k]["FID"])
best_ssim = max(results, key=lambda k: results[k]["SSIM"])

print("\n=== MELHOR MODELO POR CRITÉRIO ===")
print(f"  IS   mais alto : {best_is}")
print(f"  FID  mais baixo: {best_fid}")
print(f"  SSIM mais alto : {best_ssim}")

votes  = Counter([best_is, best_fid, best_ssim])
winner = votes.most_common(1)[0][0]
print(f"\n🏆  VENCEDOR (votação 3 critérios): {winner}")